# P00b — DOSE Subnational GDP

Load and explore the DOSE (Database of Subnational Economic output). Convert to Arrow for downstream use.

**Source:** [DOSE V2.11](https://zenodo.org/records/16313760) — 46,851 region-year rows, 83 countries, 1,661 regions, 1953–2020. Includes sectoral breakdown (agriculture, manufacturing, services) and climate variables (temperature, precipitation).

In [ ]:
include("phase00b/functions/load_phase00b.jl")

## 1. Load Subnational Panel

In [ ]:
sub = dose_load_subnational()

In [ ]:
describe(sub)

In [ ]:
first(sub, 5)

## 2. Coverage

In [ ]:
println("Rows:      $(nrow(sub))")
println("Years:     $(minimum(sub.year)) – $(maximum(sub.year))")
println("Countries: $(length(unique(sub.iso3)))")
println("Regions:   $(length(unique(sub.gid_1)))")
println()
# Regions per country
reg_per_country = combine(groupby(sub, :iso3), :gid_1 => (x -> length(unique(x))) => :n_regions)
sort!(reg_per_country, :n_regions, rev=true)
println("Top 10 by region count:")
first(reg_per_country, 10)

In [ ]:
# Temporal coverage — years per country
yr_per_country = combine(groupby(sub, :iso3), :year => minimum => :first_year, :year => maximum => :last_year,
                         :year => length => :n_obs)
sort!(yr_per_country, :first_year)
println("Earliest start: $(minimum(yr_per_country.first_year))")
println("Latest end:     $(maximum(yr_per_country.last_year))")
println()
# Countries starting before 1960
println("Countries with data before 1960: $(count(yr_per_country.first_year .< 1960))")
println("Countries with data from 1990+:  $(count(yr_per_country.first_year .>= 1990))")

## 3. Missingness — Sectoral and Climate

In [ ]:
# Completeness of each column
for col in names(sub)
    n_present = count(!ismissing, sub[!, col])
    pct = round(100 * n_present / nrow(sub), digits=1)
    println("  $(rpad(col, 25)) $(n_present) / $(nrow(sub))  ($pct%)")
end

## 4. Aggregate to Country-Year

In [ ]:
using Statistics
nat = dose_aggregate_national(sub)
describe(nat)

In [ ]:
# Spot check: compare a few countries
filter(r -> r.iso3 == "USA" && r.year == 2015, nat)

## 5. Subnational Dispersion (Preview for Signatures 3 & 4)

Quick look at within-country GDP dispersion — the foundation for scale invariance and fractal structure testing.

In [ ]:
# Within-country coefficient of variation of GDP per capita (2015)
yr2015 = filter(r -> r.year == 2015 && !ismissing(r.gdp_pc_usd2015), sub)
dispersion = combine(groupby(yr2015, [:iso3, :country]),
    :gdp_pc_usd2015 => std => :sd_gdp,
    :gdp_pc_usd2015 => mean => :mean_gdp,
    nrow => :n_regions
)
dispersion[!, :cv] = dispersion.sd_gdp ./ dispersion.mean_gdp
filter!(r -> r.n_regions >= 3, dispersion)  # need at least 3 regions for meaningful CV
sort!(dispersion, :cv, rev=true)
println("Top 15 most internally unequal (by CV of subnational GDP pc, 2015):")
first(select(dispersion, :iso3, :country, :n_regions, :mean_gdp, :cv), 15)

## 6. Write Arrow

In [ ]:
Arrow.write(PATH_DOSE_SUBNATIONAL_ARROW, sub)
println("  → $(PATH_DOSE_SUBNATIONAL_ARROW)  ($(round(filesize(PATH_DOSE_SUBNATIONAL_ARROW) / 1024^2, digits=1)) MB)")

Arrow.write(PATH_DOSE_NATIONAL_ARROW, nat)
println("  → $(PATH_DOSE_NATIONAL_ARROW)  ($(round(filesize(PATH_DOSE_NATIONAL_ARROW) / 1024^2, digits=1)) MB)")